# Optimizacion de decisiones de compra - NovaPlas Hogar
### Version simple (Google Colab)

Responde la pregunta del caso: **que productos comprar, en que cantidad,
y si conviene en costo y presupuesto?**

Cada bloque de codigo de este notebook se usa directamente para llegar a
la solucion final (no hay demostraciones sueltas).


## 1. Librerias

- `pandas` (`pd`): leer los CSV y trabajar con tablas.
- `numpy` (`np`): calculos numericos sobre arreglos.
- `matplotlib.pyplot` (`plt`): el grafico final.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Cargar los 5 archivos CSV desde GitHub

Los 5 CSV están en el repositorio de GitHub, dentro de la carpeta `data/`.
`pd.read_csv` puede leer un archivo directamente desde una URL, así que no
hace falta subir nada a mano: solo apuntamos a la versión **raw** de cada
archivo (el botón "Raw" que aparece al abrir un archivo en GitHub).

Reemplaza `usuario_github` y `nombre_repositorio` por los tuyos (los ves
en la URL del repositorio: `github.com/usuario_github/nombre_repositorio`).
Si tus CSV no están dentro de una carpeta `data/` sino en la raíz del
repositorio, quita `"data/"` de `url_base`.


In [ ]:
usuario_github = "tu-usuario"
nombre_repositorio = "tu-repositorio"
rama = "main"

url_base = f"https://raw.githubusercontent.com/LeidyV96/ProyectoPAD_Analisis_Compras/refs/heads/main/parametros_generales = pd.read_csv(url_base + "parametros_generales.csv")
tiempos_entrega = pd.read_csv(url_base + "tiempos_entrega.csv")
base = pd.read_csv(url_base + "base_productos.csv")
ventas = pd.read_csv(url_base + "historico_ventas_2025.csv")
inventario = pd.read_csv(url_base + "inventario_actual.csv")

base.head()


## 3. Parametros del negocio y tiempos de entrega

Los parametros (dias del mes, limite de costo, presupuesto) se leen del
CSV. El tiempo de entrega depende del proveedor, asi que armamos un
**diccionario** `{proveedor: tiempo_de_entrega}` con `dict()` y `zip()`:
esto nos deja "buscar" el tiempo de entrega de cualquier producto
conociendo solo su proveedor.


In [ ]:
dias_mes = int(parametros_generales.loc[parametros_generales["Parametro"] == "Días del mes", "Valor"].iloc[0])
limite_costo = float(parametros_generales.loc[parametros_generales["Parametro"] == "Límite aumento del costo", "Valor"].iloc[0])
presupuesto = float(parametros_generales.loc[parametros_generales["Parametro"] == "Presupuesto de compras", "Valor"].iloc[0])

dict_tiempos_entrega = dict(zip(tiempos_entrega["Proveedor"], tiempos_entrega["Tiempo de entrega"]))

print(f"Dias del mes: {dias_mes}")
print(f"Limite de aumento de costo: {limite_costo:.0%}")
print(f"Presupuesto: ${presupuesto:,.0f}")
print(f"Tiempos de entrega: {dict_tiempos_entrega}")


## 4. Limpiar el inventario

Algunas filas de `Inventario Actual` traen el texto `"#N/D"` en vez de un
numero. Con `pd.to_numeric(..., errors="coerce")` esos casos quedan como
`NaN` (vacio); con una **mascara booleana** (`.notna()`) identificamos
cuales son validos, y con `df.loc[mascara].copy()` nos quedamos solo con
esas filas.


In [ ]:
inventario_actual_num = pd.to_numeric(inventario["Inventario Actual"], errors="coerce")
mascara_inventario_valido = inventario_actual_num.notna()

inventario_valido = inventario.loc[mascara_inventario_valido].copy()
inventario_valido["Inventario Actual"] = inventario_actual_num.loc[mascara_inventario_valido]
inventario_valido["Pedidos pendientes"] = pd.to_numeric(
    inventario_valido["Pedidos pendientes"], errors="coerce"
).fillna(0)
inventario_valido["Stock de seguridad"] = pd.to_numeric(
    inventario_valido["Stock de seguridad"], errors="coerce"
)

print(f"Productos con inventario valido: {mascara_inventario_valido.sum()}")
print(f"Productos excluidos (dato invalido): {(~mascara_inventario_valido).sum()}")


## 5. Venta mensual promedio de cada producto

Tomamos las 12 columnas de meses, las convertimos a un array de NumPy con
`.to_numpy()` (queda una matriz de 1000 productos x 12 meses) y calculamos
el promedio **por fila** (`axis=1`): un promedio por producto, recorriendo
sus 12 meses.


In [ ]:
columnas_meses = [c for c in ventas.columns if c not in ("SKU", "Descripcion")]
matriz_ventas = ventas[columnas_meses].to_numpy()

venta_mensual = matriz_ventas.mean(axis=1)   # promedio por producto (por fila)
print("Venta mensual promedio (primeros 5 productos):", venta_mensual[:5])


## 6. Unir todo en una sola tabla

In [ ]:
tabla = base.merge(
    pd.DataFrame({"SKU": ventas["SKU"], "venta_mensual": venta_mensual}),
    on="SKU",
)
tabla = tabla.merge(
    inventario_valido[["SKU", "Inventario Actual", "Pedidos pendientes", "Stock de seguridad", "Costo actual"]],
    on="SKU",
)
tabla["tiempo_entrega"] = tabla["Proveedor"].map(dict_tiempos_entrega)

print(f"SKU en la tabla final: {len(tabla)} de {len(base)}")
tabla.head()


## 7. Corregir Stock de seguridad atipico

Si el stock de seguridad de un producto representa **mas de 300 veces su
demanda** durante el tiempo de entrega, lo tratamos como un dato erroneo y
lo llevamos a 0 (igual que a un producto que nunca tuvo ese dato). Se
calcula con `np.where` y una mascara booleana.


In [ ]:
venta_diaria = tabla["venta_mensual"].to_numpy() / dias_mes
demanda = venta_diaria * tabla["tiempo_entrega"].to_numpy()
stock_seguridad = tabla["Stock de seguridad"].to_numpy()

razon_stock_demanda = np.zeros(len(tabla))
mascara_demanda_positiva = demanda > 0
razon_stock_demanda[mascara_demanda_positiva] = (
    stock_seguridad[mascara_demanda_positiva] / demanda[mascara_demanda_positiva]
)

mascara_stock_atipico = razon_stock_demanda > 300
print("SKU con Stock de seguridad atipico (se corrigen a 0):", mascara_stock_atipico.sum())

stock_seguridad_corregido = np.where(mascara_stock_atipico, 0, stock_seguridad)
stock_seguridad_corregido = np.where(np.isnan(stock_seguridad_corregido), 0, stock_seguridad_corregido)
tabla["Stock de seguridad"] = stock_seguridad_corregido


## 8. Calcular necesidad de compra y conveniencia de costo

Dos funciones cortas con las formulas del caso, aplicadas a todo el
portafolio de una vez con arrays. Luego, una **mascara booleana con dos
condiciones a la vez (`&`)**: un producto se recomienda solo si necesita
compra **y** es conveniente en costo.


In [ ]:
def diferencia_porcentual_costo(costo_actual, costo_promedio):
    """Cuanto por ciento mas caro esta el costo actual frente al promedio."""
    return (costo_actual - costo_promedio) / costo_promedio * 100


def cantidad_en_empaques(cantidad_necesaria, unidad_empaque):
    """Redondea hacia arriba una cantidad al multiplo de empaque mas cercano."""
    return np.ceil(cantidad_necesaria / unidad_empaque) * unidad_empaque


In [ ]:
inventario_actual_arr = tabla["Inventario Actual"].to_numpy()
pedidos_arr = tabla["Pedidos pendientes"].to_numpy()
stock_seguridad_arr = tabla["Stock de seguridad"].to_numpy()
costo_actual_arr = tabla["Costo actual"].to_numpy()
costo_promedio_arr = tabla["Costo Promedio"].to_numpy()
empaque_arr = tabla["Unidad de Empaque"].to_numpy()

punto_reorden = demanda + stock_seguridad_arr
inv_proyectado = inventario_actual_arr + pedidos_arr
dif_inventario = punto_reorden - inv_proyectado
dif_porcentual = diferencia_porcentual_costo(costo_actual_arr, costo_promedio_arr)

mascara_necesita_compra = dif_inventario > 0
mascara_costo_conveniente = dif_porcentual <= (limite_costo * 100)
mascara_recomendado = mascara_necesita_compra & mascara_costo_conveniente

print("Necesitan compra:", mascara_necesita_compra.sum())
print("Recomendados (necesita compra y es conveniente):", mascara_recomendado.sum())


In [ ]:
# np.zeros: reservamos el array de cantidad sugerida en 0,
# y solo lo llenamos donde si se recomienda comprar
cantidad_sugerida = np.zeros(len(tabla))
cantidad_sugerida[mascara_recomendado] = cantidad_en_empaques(
    dif_inventario[mascara_recomendado], empaque_arr[mascara_recomendado]
)
costo_compra = cantidad_sugerida * costo_actual_arr

tabla["dif_inventario"] = dif_inventario
tabla["punto_reorden"] = punto_reorden
tabla["cant_sugerida"] = cantidad_sugerida
tabla["costo_compra"] = costo_compra
tabla["Estado_compra"] = np.where(mascara_recomendado, "Comprar", "No comprar")

print(f"Costo total si se compra todo lo recomendado: ${costo_compra[mascara_recomendado].sum():,.0f}")


## 9. Priorizar por presupuesto

Con `df.loc[mascara].copy()` nos quedamos solo con los productos
recomendados. La **urgencia** de cada uno es que tan grande es su faltante
frente a su propio punto de reorden; ordenamos de mas a menos urgente, y
vamos sumando el costo (`.cumsum()`) hasta donde alcance el presupuesto.


In [ ]:
columnas_reporte = ("SKU", "Descripcion", "Proveedor", "cant_sugerida", "costo_compra")

recomendados = tabla.loc[mascara_recomendado].copy()
recomendados["urgencia"] = np.where(
    recomendados["punto_reorden"] > 0,
    recomendados["dif_inventario"] / recomendados["punto_reorden"],
    0,
)
recomendados = recomendados.sort_values("urgencia", ascending=False)
recomendados["costo_acumulado"] = recomendados["costo_compra"].cumsum()

mascara_aprobado = recomendados["costo_acumulado"] <= presupuesto
aprobados = recomendados.loc[mascara_aprobado].copy()

print(f"Productos recomendados en total: {len(recomendados)}")
print(f"Productos aprobados dentro del presupuesto: {len(aprobados)}")
print(f"Costo total aprobado: ${aprobados['costo_compra'].sum():,.0f}")

aprobados[list(columnas_reporte)].head(10)


## 10. Resultados agregados

In [ ]:
costos_aprobados = aprobados["costo_compra"].to_numpy()

print("Suma total aprobada:  ", f"${costos_aprobados.sum():,.0f}")
print("Promedio por producto:", f"${costos_aprobados.mean():,.0f}")
print("Compra mas grande:    ", f"${costos_aprobados.max():,.0f}")
print("Desviacion estandar:  ", f"${costos_aprobados.std():,.0f}")

resumen_proveedor = aprobados.groupby("Proveedor")["costo_compra"].sum().sort_values(ascending=False)
print("\nTotal aprobado por proveedor:")
print(resumen_proveedor)


## 11. Grafico de torta: compra aprobada por proveedor

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    resumen_proveedor.values,
    labels=resumen_proveedor.index,
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Distribucion de la compra aprobada por proveedor")
ax.axis("equal")
plt.tight_layout()
plt.savefig("distribucion_compra_por_proveedor.png", dpi=150)
plt.show()


## 12. Exportar la solucion a Excel y descargarla

Tres hojas: la compra aprobada, el resumen por proveedor, y la tabla
completa con el estado de cada SKU.

In [ ]:
nombre_archivo = "solucion_compras_novaplas.xlsx"

with pd.ExcelWriter(nombre_archivo, engine="openpyxl") as writer:
    aprobados[list(columnas_reporte) + ["urgencia", "costo_acumulado"]].to_excel(
        writer, sheet_name="Compra aprobada", index=False
    )
    resumen_proveedor.reset_index().rename(columns={"costo_compra": "Costo total aprobado"}).to_excel(
        writer, sheet_name="Resumen por proveedor", index=False
    )
    tabla[["SKU", "Descripcion", "Proveedor", "Inventario Actual", "punto_reorden",
           "dif_inventario", "cant_sugerida", "costo_compra", "Estado_compra"]].to_excel(
        writer, sheet_name="Todos los productos", index=False
    )

print(f"Archivo generado: {nombre_archivo}")

try:
    from google.colab import files
    files.download(nombre_archivo)
except ImportError:
    print("Fuera de Colab: el archivo quedo guardado en la carpeta actual del notebook.")
